### Step 3 - Novelty Scoring (Unsupervised, Temporal) - Feature Construction Stage

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

- STEP 3B — Citation Features
- Extract:
- cited_by_count
- reference_count

In [2]:
meta = pd.read_csv("../../outputs/intermediate/openalex_metadata_full.csv")

citation_df = meta[[
    "global_paper_id",
    "cited_by_count",
    "referenced_works",
    "year"
]].copy()


def count_refs(x):
    if pd.isna(x):
        return 0
    try:
        return len(eval(x))
    except:
        return 0


citation_df["reference_count"] = citation_df["referenced_works"].apply(count_refs)

citation_df = citation_df.rename(columns={
    "global_paper_id": "paper_id"
})

citation_df = citation_df[[
    "paper_id",
    "year",
    "cited_by_count",
    "reference_count"
]]

citation_df.to_csv("../../outputs/other/citation_features.csv", index=False)

print("Citation features saved.")

Citation features saved.


- STEP 3C — Feature Matrix Construction
- Combine:
- Semantic novelty (kNN)
- Structural novelty
- Citation features

In [3]:
semantic_df = pd.read_csv("../../outputs/final/semantic_novelty_knn_scores.csv")
struct_df = pd.read_csv("../../outputs/final/structural_novelty_scores.csv")
citation_df = pd.read_csv("../../outputs/other/citation_features.csv")

# Merge
features = semantic_df.merge(struct_df,
                             on="paper_id",
                             how="left")

features = features.merge(citation_df,
                          on=["paper_id", "year"],
                          how="left")

features = features.fillna(0)

print("Feature matrix shape:", features.shape)

features.to_csv("../../outputs/other/novelty_feature_matrix.csv", index=False)

Feature matrix shape: (2511, 7)


- STEP 3D — Composite Novelty Score (Continuous)
- Instead of classification, we define:
-   composite_novelty =
-       w1 * structural +
-       w2 * semantic +
-       w3 * citation_signal
- Citation signal normalized.

In [4]:
features = pd.read_csv("../../outputs/other/novelty_feature_matrix.csv")

# Normalize citation signal (log transform for stability)
features["citation_signal"] = np.log1p(features["cited_by_count"])


# Min-max normalize components
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


features["struct_norm"] = minmax(features["structural_novelty"])
features["semantic_norm"] = minmax(features["semantic_knn"])
features["citation_norm"] = minmax(features["citation_signal"])

# Weighted combination (can tune later)
features["composite_novelty"] = (
        0.20 * features["struct_norm"] +
        0.70 * features["semantic_norm"] +
        0.10 * features["citation_norm"]
)

features.to_csv("../../outputs/final/novelty_feature_matrix_with_score.csv",
                index=False)

print("Composite novelty score computed.")

Composite novelty score computed.


In [5]:
pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv').columns

Index(['paper_id', 'year', 'semantic_knn', 'structural_novelty',
       'triple_count', 'cited_by_count', 'reference_count', 'citation_signal',
       'struct_norm', 'semantic_norm', 'citation_norm', 'composite_novelty'],
      dtype='object')

In [6]:
fm = pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv')
fm.head()

,paper_id,year,semantic_knn,structural_novelty,triple_count,cited_by_count,reference_count,citation_signal,struct_norm,semantic_norm,citation_norm,composite_novelty
0,SKG_SUM_7,2010,1.0,1.0,336,20,26,3.044522,1.0,1.0,0.336536,0.933654
1,SKG_MT_1325,2010,1.0,1.0,107,34,21,3.555348,1.0,1.0,0.393002,0.939300
2,SKG_MT_1231,2010,1.0,1.0,59,4,14,1.609438,1.0,1.0,0.177904,0.917790
3,SKG_MT_1232,2010,1.0,1.0,95,14,13,2.708050,1.0,1.0,0.299343,0.929934
4,SKG_MT_388,2010,1.0,1.0,61,32,29,3.496508,1.0,1.0,0.386498,0.938650


In [7]:
features[['semantic_knn', 'structural_novelty']].corr()

,semantic_knn,structural_novelty
semantic_knn,1.000000,-0.094759
structural_novelty,-0.094759,1.000000


In [8]:
df_clean = fm[fm['year']>2010]
df_clean = df_clean[df_clean['triple_count']>0]
df_clean.head()

,paper_id,year,semantic_knn,structural_novelty,triple_count,cited_by_count,reference_count,citation_signal,struct_norm,semantic_norm,citation_norm,composite_novelty
59,SKG_SUM_45,2011,0.902470,0.913043,69,39,22,3.688879,0.913043,0.772936,0.407762,0.764440
60,SKG_PAR_94,2011,0.924697,0.898305,59,25,37,3.258097,0.898305,0.824683,0.360144,0.792953
61,SKG_SUM_80,2011,0.919655,0.934579,107,76,26,4.343805,0.934579,0.812945,0.480157,0.803993
62,SKG_SA_505,2011,0.919154,0.906667,75,164,31,5.105945,0.906667,0.811779,0.564402,0.806019
63,SKG_MT_620,2011,0.922090,1.000000,2,6,17,1.945910,1.000000,0.818615,0.215097,0.794540


In [9]:
df_clean.groupby('year').size()

year
2011     64
2012     73
2013    122
2014     89
2015    116
2016    129
2017    221
2018    290
2019    436
2020    504
2021    403
2022      3
2024      1
2025      1
dtype: int64

In [10]:
from scipy import stats
r, p = stats.pearsonr(df_clean['semantic_knn'], df_clean['structural_novelty'])

In [11]:
print(f"Clean correlation: r={r:.4f}, p={p:.4f}, n={len(df_clean)}")

Clean correlation: r=-0.2715, p=0.0000, n=2452


In [12]:
pn = pd.read_csv('../../outputs/final/paper_nodes.csv')
pn.head()

,node_id,node_type,title,domain,split,year,openalex_id,cited_by_count,score,method
0,NOVEL_DIA_0,Paper,MRF-Chat Improving Dialogue with Markov Random...,DIA,NOVEL,2021.0,https://openalex.org/W3214342458,0.0,0.606481,tfidf
1,NOVEL_DIA_1,Paper,Towards Making the Most of Dialogue Characteri...,DIA,NOVEL,2021.0,https://openalex.org/W3196896228,12.0,0.663980,tfidf
2,NOVEL_DIA_2,Paper,Domain-Adaptive Pretraining Methods for Dialog...,DIA,NOVEL,2021.0,https://openalex.org/W3173606101,19.0,0.668112,tfidf
3,NOVEL_DIA_3,Paper,Adaptive Bridge between Training and Inference...,DIA,NOVEL,2021.0,https://openalex.org/W3214623240,5.0,0.636847,tfidf
4,NOVEL_DIA_4,Paper,Controlling Dialogue Generation with Semantic ...,DIA,NOVEL,2021.0,https://openalex.org/W3074476581,6.0,0.811583,tfidf


In [13]:
df = fm.merge(pn[['node_id','split','domain']], left_on='paper_id', right_on='node_id', how='left')
# df = df[df['split'] == 'NOVEL']
# df.set_index('split')
df.groupby('split')[['struct_norm', 'semantic_norm', 'composite_novelty']].mean()

,struct_norm,semantic_norm,composite_novelty
split,,,
NOVEL,0.708848,0.892055,0.794585
SKG,0.769077,0.856172,0.789185
